In [ ]:
# EDA — HackYeah sessions

This notebook explores session data from `data/sessions`.

The aim is to inspect dataset structure, data quality and key feature distributions, then prepare data for further analysis or modeling.

## Data source

- session directory: `/workspace/hackaton/HackYeach-2026-REPO/data/sessions`
- session structure: `meta.json`, `events.jsonl`, `face.jsonl`, `net.jsonl`, `video/*`
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

ROOT = Path("/workspace/hackaton/HackYeach-2026-REPO")
DATA_DIR = ROOT / "data" / "sessions"

print(f"ROOT: {ROOT}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"Directory exists: {DATA_DIR.exists()}")
print(f"Subdirectory count: {len(list(DATA_DIR.iterdir())) if DATA_DIR.exists() else 0}")

session_dirs = sorted([p for p in DATA_DIR.iterdir() if p.is_dir()])
print(f"Session count: {len(session_dirs)}")
print("First 5 sessions:")
for p in session_dirs[:5]:
    print(" -", p.name)
# Load session data

rows = []
for session_dir in session_dirs:
    meta_path = session_dir / "meta.json"
    if not meta_path.exists():
        continue
    try:
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
    except Exception:
        meta = {}

    events_path = session_dir / "events.jsonl"
    face_path = session_dir / "face.jsonl"
    net_path = session_dir / "net.jsonl"
    video_dir = session_dir / "video"

    row = {
        "session_id": meta.get("session_id") or session_dir.name,
        "participant_code": meta.get("participant_code"),
        "condition": meta.get("condition"),
        "order_index": meta.get("order_index"),
        "started_at": meta.get("started_at"),
        "ended_at": meta.get("ended_at"),
        "has_events": events_path.exists(),
        "has_face": face_path.exists(),
        "has_net": net_path.exists(),
        "video_files": len(list(video_dir.glob("*"))) if video_dir.exists() else 0,
        "survey_present": (session_dir / "survey.json").exists(),
    }
    rows.append(row)

sessions_df = pd.DataFrame(rows)
print(sessions_df.head())
print("\nColumn types:")
print(sessions_df.dtypes)

if sessions_df.empty:
    raise ValueError("No sessions to analyze.")
# Dataset structure overview

print(f"Row count: {sessions_df.shape[0]}")
print(f"Column count: {sessions_df.shape[1]}")
print("\nColumn names:")
print(list(sessions_df.columns))
print("\nBasic information:")
sessions_df.info()

sessions_df.describe(include=["number"]).T
# Assess missing data and duplicates

missing = sessions_df.isna().sum().sort_values(ascending=False)
print("Missing data:")
print(missing[missing > 0])

print("\nDuplicates:")
print(f"Duplicate record count: {sessions_df.duplicated().sum()}")

# missing-data chart
plt.figure(figsize=(10, 5))
missing.plot(kind="bar", color="steelblue")
plt.title("Missing data by column")
plt.ylabel("Missing value count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
# Descriptive statistics

numeric_cols = [c for c in sessions_df.columns if pd.api.types.is_numeric_dtype(sessions_df[c])]
print("Numeric columns:")
print(numeric_cols)

if numeric_cols:
    print(sessions_df[numeric_cols].describe().T)

categorical_cols = [c for c in sessions_df.columns if c not in numeric_cols]
print("\nCategorical columns:")
print(categorical_cols)

if categorical_cols:
    print(sessions_df[categorical_cols].describe(include=["object", "category"]).T)
# Numeric variable distributions

# prepare additional helper variables
if "order_index" in sessions_df.columns:
    plt.figure(figsize=(10, 5))
    sns.histplot(sessions_df["order_index"].dropna(), bins=20, kde=True)
    plt.title("Session order distribution")
    plt.xlabel("order_index")
    plt.tight_layout()
    plt.show()

if "video_files" in sessions_df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=sessions_df["video_files"], color="lightblue")
    plt.title("Video files per session: box plot")
    plt.xlabel("video file count")
    plt.tight_layout()
    plt.show()

if "started_at" in sessions_df.columns:
    started = pd.to_numeric(sessions_df["started_at"], errors="coerce").dropna()
    if not started.empty:
        plt.figure(figsize=(10, 5))
        sns.histplot(started, bins=30, kde=True)
        plt.title("Session start-time distribution")
        plt.xlabel("started_at (timestamp)")
        plt.tight_layout()
        plt.show()
# Categorical variable analysis

if "condition" in sessions_df.columns:
    cond_counts = sessions_df["condition"].value_counts(dropna=False)
    print("Condition counts:")
    print(cond_counts)

    plt.figure(figsize=(10, 5))
    sns.countplot(data=sessions_df, x="condition", order=cond_counts.index)
    plt.title("Sessions by condition")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

if "participant_code" in sessions_df.columns:
    print("\nMost frequent participant codes:")
    print(sessions_df["participant_code"].value_counts().head())
# Correlations and relationships between variables

corr_df = sessions_df.select_dtypes(include=[np.number])
if not corr_df.empty:
    corr = corr_df.corr(numeric_only=True)
    print("Correlation matrix:")
    print(corr)

    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Numeric variable correlation matrix")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric variables for correlation analysis.")
# Visualize trends and outliers

if {"order_index", "video_files"}.issubset(sessions_df.columns):
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=sessions_df, x="order_index", y="video_files", hue="condition", s=80)
    plt.title("Video file count by session order")
    plt.xlabel("order_index")
    plt.ylabel("video_files")
    plt.tight_layout()
    plt.show()

if "condition" in sessions_df.columns and "video_files" in sessions_df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=sessions_df, x="condition", y="video_files")
    plt.title("Video file count distribution by condition")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()
# Prepare data for further analysis or modeling

# Apply a simple, practical data preparation plan
# 1. remove duplicates
# 2. fill missing values in relevant columns
# 3. normalize data types
# 4. encode categorical columns
# 5. handle outliers

clean_df = sessions_df.copy()
clean_df = clean_df.drop_duplicates()

for col in ["order_index", "video_files"]:
    if col in clean_df.columns:
        clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

for col in ["condition", "participant_code"]:
    if col in clean_df.columns:
        clean_df[col] = clean_df[col].fillna("unknown")

print("Prepared dataset:")
print(clean_df.head())
print("\nRecord count after removing duplicates:", clean_df.shape[0])

# example encoding of categorical columns
if "condition" in clean_df.columns:
    encoded = pd.get_dummies(clean_df[["condition"]], prefix=["condition"])
    print("\nCategory encoding example:")
    print(encoded.head())

print("\nFurther data preparation steps:")
print("1. Fill missing values in columns relevant to the analysis.")
print("2. Encode categorical variables using one-hot or ordinal encoding.")
print("3. Normalize or standardize numeric columns before modeling.")
print("4. Review outliers and their potential effect on the model.")
print("5. Consider extracting features from `events.jsonl`, `face.jsonl` and `net.jsonl` into per-session tables.")